<a href="https://colab.research.google.com/github/stauntonjr/local_llm_notebooks/blob/master/2_bit_QLoRA_with_Qwen3_and_with_EoRA_Initialization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*More details in this article: [Fine-Tuning 2-Bit Qwen3 Models on Your Computer](https://kaitchup.substack.com/p/fine-tuning-2-bit-qwen3-models-on)*


This notebook demonstrates how to fine-tune large language models quantized in the **GPTQ** format. It supports models quantized with:

- **AutoRound**
- **AutoGPTQ**
- **GPTQModel**

Tested Configuration:

- **Model**: Qwen3 (14B)  
- **Precision**: 2-bit GPTQ  
- **Adapter**: Can be initialized from scratch or loaded from a pre-existing one (e.g., calibrated with **EoRA**)

System Requirements:

- Fine-tuning a **Qwen3-14B** model quantized to **2-bit** is feasible on a **single 16 GB GPU**.

> ✅ You can easily adapt this workflow to other GPTQ-quantized models.


# Fine-Tuning Code

In [ ]:
import torch, os, multiprocessing
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training, PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed
)
from trl import SFTTrainer, SFTConfig
set_seed(1234)


def fine_tune(model_name, batch_size=1, gradient_accumulation_steps=32, EoRA=False):

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    ds_train = load_dataset("allenai/tulu-3-sft-mixture", split="train[:150000]")
    def process(row):
        row["text"] = tokenizer.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False, enable_thinking=False)
        return row

    ds_train = ds_train.map(
        process,
        num_proc= multiprocessing.cpu_count(),
        load_from_cache_file=False,
    )

    ds_train = ds_train.remove_columns(["messages"])

    model = AutoModelForCausalLM.from_pretrained(
            model_name, device_map={"": 0}, torch_dtype=torch.float16, trust_remote_code=True, attn_implementation="flash_attention_2"
    )
    model = prepare_model_for_kbit_training(model, gradient_checkpointing_kwargs={'use_reentrant':True})
    if EoRA:
      model = PeftModel.from_pretrained(model, "kaitchup/Qwen3-14B-autoround-2bit-gptq-EoRA-r32", is_trainable=True)
      peft_config = None
    else:
      peft_config = LoraConfig(
            lora_alpha=32,
            lora_dropout=0.0,
            r=32,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules= ['k_proj', 'q_proj', 'v_proj', 'o_proj', "gate_proj", "down_proj", "up_proj"],
      )

    name = model_name.split("/")[-1]

    output_dir = "./LoRA"+name+"_eora/"

    training_arguments = SFTConfig(
          output_dir=output_dir,
          optim="paged_adamw_8bit",
          per_device_train_batch_size=batch_size,
          gradient_accumulation_steps=gradient_accumulation_steps,
          log_level="debug",
          save_steps=100,
          logging_steps=25,
          learning_rate=1e-5,
          bf16 = True,
          max_steps=1000,
          warmup_ratio=0.03,
          lr_scheduler_type="linear",
          dataset_text_field="text",
          max_seq_length=4096,
          report_to="none",
    )

    trainer = SFTTrainer(
          model=model,
          train_dataset=ds_train,
          peft_config=peft_config,
          processing_class=tokenizer,
          args=training_arguments,
    )

    #--code by Unsloth: https://colab.research.google.com/drive/1Ys44kVvmeZtnICzWz0xgpRnrIOjZAuxp?usp=sharing#scrollTo=pCqnaKmlO1U9

    gpu_stats = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
    print(f"{start_gpu_memory} GB of memory reserved.")

    trainer_ = trainer.train()


    used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    used_memory_for_trainer= round(used_memory - start_gpu_memory, 3)
    used_percentage = round(used_memory         /max_memory*100, 3)
    trainer_percentage = round(used_memory_for_trainer/max_memory*100, 3)
    print(f"{trainer_.metrics['train_runtime']} seconds used for training.")
    print(f"{round(trainer_.metrics['train_runtime']/60, 2)} minutes used for training.")
    print(f"Peak reserved memory = {used_memory} GB.")
    print(f"Peak reserved memory for training = {used_memory_for_trainer} GB.")
    print(f"Peak reserved memory % of max memory = {used_percentage} %.")
    print(f"Peak reserved memory for training % of max memory = {trainer_percentage} %.")
    print("-----")
    #----



## QLoRA from Scratch

In [ ]:
fine_tune("kaitchup/Qwen3-14B-autoround-2bit-gptq", batch_size=1, gradient_accumulation_steps=128, EoRA=False)

## QLoRA from EoRA

In [ ]:
fine_tune("kaitchup/Qwen3-14B-autoround-2bit-gptq", batch_size=1, gradient_accumulation_steps=128, EoRA=True)